# 002: Understanding the Framework Layer (V1 Pattern G)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import ibis

from earlysign.core.ledger import Ledger

connection = ibis.connect("duckdb://:memory:")
ledger = Ledger(connection, "example_v1").bind(exp_id="exp_v1_001")
ledger.ensure()
ledger

## Tier 0: Ingesting Raw Evidence

In V1 (Pattern G), we use `Session` to define the scientific horizon and `Ingest` to record raw evidence.

In [ ]:
# Direct sess.Commit for raw evidence
from pydantic import BaseModel

from earlysign.v1.framework.session import Session


class Observation(BaseModel):
    value: float
    arm: str


with Session(ledger) as sess:
    sess.Commit(Observation(value=10.0, arm="C"), trace=[])
    sess.Commit(Observation(value=12.0, arm="T"), trace=[])

    ledger.show()

## Tier 1: State Reconstruction (Projectors)

To read data, we use `Projectors` within a `Session`. Projectors translate raw events into structured scientific state while tracking provenance.

In [ ]:
from earlysign.v1.framework.projector import ProjectionResult, Projector
from earlysign.v1.framework.trace import TraceId


class MeanProjector(Projector[dict]):
    def project(self, table: ibis.Expr) -> ProjectionResult[dict]:
        matched = table.filter(table.type == "Observation")
        pdf = matched.execute()
        if pdf.empty:
            return ProjectionResult(data={"mean": 0.0}, trace=[])

        mean_val = pdf["payload"].apply(lambda x: x["value"]).mean()
        # Extract uuids as trace
        traces = [TraceId(str(u)) for u in pdf["uuid"].tolist()]
        return ProjectionResult(data={"mean": mean_val}, trace=traces)


with Session(ledger) as sess:
    traced_mean = sess.Read(MeanProjector())
    print(f"Hydrated State: {traced_mean.data}")
    print(f"Implicit Trace (accumulated by Read): {sess.trace}")

## Tier 2: Analytical Operations (CallAndCommit)

We can perform analytical operations and commit their results. The lineage is automatically tracked from the inputs.

In [ ]:
from earlysign.v1.framework.writer import Writer


class AnalysisResult(BaseModel):
    doubled_mean: float


def analyze_mean(summary: dict):
    return {"doubled_mean": summary["mean"] * 2}


with Session(ledger) as sess:
    traced_mean = sess.Read(MeanProjector())

    Writer.CallAndCommit(sess, AnalysisResult, analyze_mean, summary=traced_mean)
    print("Result committed.")

    ledger.show()

# Multi-Step Orchestration: Ledger Concept via Template API

We now demonstrate the procedure of a group sequential test using the high-level Template API.
This includes performing updates and explicitly calling specialized Progress and Final reports.

In [ ]:
import numpy as np

from earlysign.schema.ES3.Binomial import ArmData
from earlysign.v1.methods.group_sequential.plan.protocol_design import ProtocolDesigner
from earlysign.v1.templates.binomial_ab import BinomialABTemplate

# Clear and bind a new experiment
ledger = Ledger(ibis.connect("duckdb://:memory:"), "events").bind(
    experiment="reporting_v1"
)
ledger.ensure()

## Step 1: Initialize Protocol and Design

We define the design and initialize the template using a unified `set_protocol` API.

In [ ]:
from earlysign.v1.templates.binomial_ab import BinomialABProtocol

planner = ProtocolDesigner.from_dict({"model": "canonical_joint"})
protocol = planner.plan_binomial_ab(
    alpha=0.05,
    power=0.8,
    p_control=0.10,
    delta=0.03,
    k=5,
)

trial = BinomialABTemplate(ledger)
protocol = BinomialABProtocol(**protocol.model_dump())
trial.set_protocol(protocol)

print(f"Planned N_max: {int(protocol.method.stopping_policy.timer.max_sample_size)}")

## Step 2: Multi-Look Simulation with On-demand Reporting

We simulate the study. We call `update()` for each batch and explicitly call report methods when a stop/look is reached.

In [ ]:
p_c, p_t = 0.10, 0.15
n_per_batch = int(protocol.method.stopping_policy.timer.max_sample_size) // 5

for look in range(1, 6):
    print(f"--- Processing Look {look} ---")
    s_c = np.random.binomial(n_per_batch, p_c)
    s_t = np.random.binomial(n_per_batch, p_t)

    batch = [
        ArmData(n=n_per_batch, success=s_c, arm="C"),
        ArmData(n=n_per_batch, success=s_t, arm="T"),
    ]

    # update() performs analysis and returns minimal status info
    trial.update(batch)
    res = trial.report_progress()
    print(f"Progress result: {res}")

    if res["look"]:
        if res["status"] == "STOP_EFFICACY":
            print(">>> STOPPED EARLY!")
            # Explicitly call report_result() at the end
            final = trial.report_result()
            print(f"Detailed Final Report: {final}")
            break
        elif look == 5:
            final = trial.report_result()
            print(f"Detailed Final Report: {final}")